In [1]:
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Config Files"
OUTPUT_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "7.2-Project_List_API_3rdAttempt_ShallowC.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "7.2-Project_List_API_Details_3rdAttempt_ShallowC.csv")


# === EXTRACT ALL API LEVELS (both matrix and hardcoded)
def extract_all_api_levels(obj):
    api_levels = set()

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = str(k).lower()

                # Match structured API fields
                if key_lower in ['api-level', 'api'] or key_lower.startswith('android-'):
                    if isinstance(v, list):
                        for val in v:
                            if str(val).isdigit():
                                api_levels.add(str(val))
                    elif isinstance(v, (int, str)) and str(v).isdigit():
                        api_levels.add(str(v))
                else:
                    recurse(v)

        elif isinstance(o, list):
            for item in o:
                recurse(item)

        elif isinstance(o, str):
            # Look for common API patterns in strings like "api 28", "targetSdk=28"
            matches = re.findall(r'\b(?:api(?:-level)?|targetSdk|compileSdk)[\s:=]*["\']?(\d{2,3})["\']?', o, flags=re.IGNORECASE)
            for match in matches:
                api_levels.add(match)

    recurse(obj)
    return api_levels


# === PARSE YAML FILE ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            content = yaml.safe_load(raw)
            if not content:
                return {'api_levels': set(), 'error': True}
            all_api_levels = extract_all_api_levels(content)
            return {'api_levels': all_api_levels, 'error': False}
    except Exception:
        return {'api_levels': set(), 'error': True}


# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            username = parts[0].lower() if len(parts) > 1 else "unknown"
            project_name = parts[1].lower() if len(parts) > 2 else parts[0].lower()
            full_name = f"{username}.{project_name}"

            result = parse_yaml_file(file_path)

            if full_name not in project_results:
                project_results[full_name] = {
                    'username': username,
                    'project_name': project_name,
                    'api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[full_name] = []

            project_results[full_name]['api_levels'].update(result['api_levels'])
            project_results[full_name]['yml_count'] += 1
            if result['error']:
                project_results[full_name]['errors'] += 1

# === BUILD DETAILED ROWS ===
final_detailed_rows = []
for full_name, data in project_results.items():
    for api in data['api_levels']:
        final_detailed_rows.append({
            'username': data['username'],
            'project_name': data['project_name'],
            'full_name': full_name,
            'api_level': api,
            'source': 'detected',
            'yml_count': data['yml_count']
        })

# === EXPORT SUMMARY CSV ===
summary_rows = []
for full_name, result in project_results.items():
    summary_rows.append({
        'username': result['username'],
        'project_name': result['project_name'],
        'full_name': full_name,
        'distinct_api_levels': len(result['api_levels']),
        'yml_count': result['yml_count'],
        'yaml_errors': result['errors']
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(final_detailed_rows).to_csv(DETAILED_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output\7.2-Project_List_API_3rdAttempt_ShallowC.csv
✅ Detailed CSV saved to: F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output\7.2-Project_List_API_Details_3rdAttempt_ShallowC.csv
